In [10]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DeiTFeatureExtractor, ViTModel
from PIL import Image
from tqdm import tqdm

In [11]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "../checkpoints/late_fusion_unixcoder/"
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

UNIXCODER_CKPT = "../checkpoints/unixcoder_only/checkpoint-2322/"
VIT_CKPT = "../checkpoints/vit_only/deit_epoch_3.pt"

TEXT_DIR = "../Text_Files/Train"
IMAGE_DIR = "../snapshots/Train"
TEST_BASE = "../snapshots"

In [12]:
from transformers import AutoModelForSequenceClassification, ViTForImageClassification, AutoImageProcessor

print("Loading fine-tuned UnixCODER...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/unixcoder-base")
text_model = AutoModelForSequenceClassification.from_pretrained(UNIXCODER_CKPT).to(DEVICE)
text_model.eval()

Loading fine-tuned UnixCODER...


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(51416, 768, padding_idx=1)
      (position_embeddings): Embedding(1026, 768, padding_idx=1)
      (token_type_embeddings): Embedding(10, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
       

In [13]:
print("Loading fine-tuned DeiT model...")
image_processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224", use_fast=True)

# Load your trained classification model first
trained_model = ViTForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=2,
    ignore_mismatched_sizes=True
)

trained_model.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
trained_model.to(DEVICE)

vit_model = trained_model
vit_model.to(DEVICE)
vit_model.eval()

Loading fine-tuned DeiT model...


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 5562.74it/s]
Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [14]:
class FusionDataset(Dataset):
    def __init__(self, text_dir, image_dir, tokenizer, image_processor):
        self.text_paths = []
        self.image_paths = []
        self.labels = []
        self.tokenizer = tokenizer
        self.image_processor = image_processor

        # Scan text folders
        for label_folder in sorted(os.listdir(text_dir)):
            label_path = os.path.join(text_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            label = int(label_folder.split("_")[1])  # e.g., "Label_0" -> 0
            for txt_file in sorted(os.listdir(label_path)):
                if txt_file.endswith(".txt"):
                    self.text_paths.append(os.path.join(label_path, txt_file))
                    self.labels.append(label)

        # Scan image folders
        self.image_paths = []
        for label_folder in sorted(os.listdir(image_dir)):
            label_path = os.path.join(image_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            for img_file in sorted(os.listdir(label_path)):
                if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(label_path, img_file))

        # Ensure text_paths and image_paths are aligned
        assert len(self.text_paths) == len(self.image_paths), "Text and image counts must match!"

    def __len__(self):
        return len(self.text_paths)

    def __getitem__(self, idx):
        # ----- TEXT -----
        with open(self.text_paths[idx], "r") as f:
            text = f.read()
        encoding = self.tokenizer(
            text, return_tensors="pt", truncation=True, padding="max_length", max_length=512
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # ----- IMAGE -----
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image_tensor = self.image_processor(images=image, return_tensors="pt")
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].squeeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_ids, attention_mask, image_tensor, label


In [15]:
dataset = FusionDataset(TEXT_DIR, IMAGE_DIR, tokenizer, image_processor)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [16]:
TEXT_BASE = "../Text_Files"
IMAGE_BASE = "../snapshots"
OUTPUT_DIR = "../checkpoints/late_fusion_unixcoder/"

In [17]:
# ---------------------- Inference Loop ----------------------
softmax = torch.nn.Softmax(dim=1)

for i in range(10):  # Test_0 ... Test_9
    print(f"\n===== Running inference on Test_{i} =====")
    text_dir = os.path.join(TEXT_BASE, f"Test_{i}")
    image_dir = os.path.join(IMAGE_BASE, f"Test_{i}")
    dataset = FusionDataset(text_dir, image_dir, tokenizer, image_processor)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    preds, gts = [], []

    with torch.no_grad():
        for input_ids, attention_mask, image_tensor, labels in tqdm(dataloader):
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            pixel_values = image_tensor["pixel_values"].to(DEVICE)
            gts.extend(labels.tolist())

            # --- unixCODER ---
            text_outputs = text_model(input_ids=input_ids, attention_mask=attention_mask)
            text_probs = softmax(text_outputs.logits)
            text_confidences, text_preds = torch.max(text_probs, dim=1)

            # --- ViT ---
            image_outputs = vit_model(pixel_values=pixel_values)
            image_probs = softmax(image_outputs.logits)
            image_confidences, image_preds = torch.max(image_probs, dim=1)

            # --- Combine ---
            combined_preds = []
            for j in range(len(text_preds)):
                if text_confidences[j] > image_confidences[j]:
                    combined_preds.append(text_preds[j].item())
                else:
                    combined_preds.append(image_preds[j].item())
            preds.extend(combined_preds)

    # Compute accuracy for this test set
    preds = torch.tensor(preds)
    gts = torch.tensor(gts)
    acc = (preds == gts).float().mean().item()
    print(f"Accuracy on Test_{i}: {acc * 100:.2f}%")


===== Running inference on Test_0 =====


100%|██████████| 126/126 [00:13<00:00,  9.44it/s]


Accuracy on Test_0: 85.03%

===== Running inference on Test_1 =====


100%|██████████| 126/126 [00:13<00:00,  9.54it/s]


Accuracy on Test_1: 81.14%

===== Running inference on Test_2 =====


...

Accuracy on Test_7: 70.56%

===== Running inference on Test_8 =====


100%|██████████| 126/126 [00:13<00:00,  9.27it/s]


Accuracy on Test_8: 77.35%

===== Running inference on Test_9 =====


100%|██████████| 126/126 [00:13<00:00,  9.68it/s]

Accuracy on Test_9: 70.56%


In [18]:
# ---------------------- Inference Loop (Average Confidence Fusion) ----------------------
softmax = torch.nn.Softmax(dim=1)

for i in range(10):  # Test_0 ... Test_9
    print(f"\n===== Running inference on Test_{i} =====")
    text_dir = os.path.join(TEXT_BASE, f"Test_{i}")
    image_dir = os.path.join(IMAGE_BASE, f"Test_{i}")
    dataset = FusionDataset(text_dir, image_dir, tokenizer, image_processor)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    preds, gts = [], []

    with torch.no_grad():
        for input_ids, attention_mask, image_tensor, labels in tqdm(dataloader):
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            pixel_values = image_tensor["pixel_values"].to(DEVICE)
            gts.extend(labels.tolist())

            # --- CodeBERT ---
            text_outputs = text_model(input_ids=input_ids, attention_mask=attention_mask)
            text_probs = softmax(text_outputs.logits)  # shape: (batch, num_labels)

            # --- ViT ---
            image_outputs = vit_model(pixel_values=pixel_values)
            image_probs = softmax(image_outputs.logits)  # shape: (batch, num_labels)

            # --- Combine by averaging probabilities ---
            combined_probs = (text_probs + image_probs) / 2.0  # element-wise average
            combined_preds = torch.argmax(combined_probs, dim=1)  # pick label with highest average prob

            preds.extend(combined_preds.cpu().tolist())

    # Compute accuracy for this test set
    preds = torch.tensor(preds)
    gts = torch.tensor(gts)
    acc = (preds == gts).float().mean().item()
    print(f"Accuracy on Test_{i}: {acc * 100:.2f}%")



===== Running inference on Test_0 =====


100%|██████████| 126/126 [00:13<00:00,  9.59it/s]


Accuracy on Test_0: 85.03%

===== Running inference on Test_1 =====


100%|██████████| 126/126 [00:13<00:00,  9.34it/s]


Accuracy on Test_1: 81.14%

===== Running inference on Test_2 =====


...

Accuracy on Test_7: 70.56%

===== Running inference on Test_8 =====


100%|██████████| 126/126 [00:13<00:00,  9.33it/s]


Accuracy on Test_8: 77.35%

===== Running inference on Test_9 =====


100%|██████████| 126/126 [00:13<00:00,  9.67it/s]

Accuracy on Test_9: 70.56%
